# SimSwap 보호 효과 견고성 실험 (FaceGuard Week 6)

Week 5의 pilot(`simswap_eval.ipynb`)을 반복 실험하기 쉽게 정리한 노트북.

**처음 한 번만:** Google Drive에 `MyDrive/faceguard/photos/` 폴더를 만들고 `person_a_1.jpg` ~ `person_a_4.jpg`를 올려 둔다.

**실행 순서:** 런타임을 T4 GPU로 설정 → `준비 1` 실행 (끝나면 런타임이 자동으로 다시 시작됨) → `준비 2` → `도구 함수` → 실험 셀

**결과 저장:** `MyDrive/faceguard/week6/<tag>/`에 바로 저장되므로 세션이 끊겨도 남는다.

출처: [SimSwap](https://github.com/neuralchen/SimSwap) (CC BY-NC 4.0, 학술·비상업 용도). 결과와 해석: `docs/weekly/week6.md`

## 준비

In [ ]:
# [준비 1] SimSwap 받기 + 패키지 설치. 끝나면 런타임이 자동으로 다시 시작된다.
!git clone -q https://github.com/neuralchen/SimSwap /content/SimSwap
!pip install -q insightface==0.2.1 onnxruntime moviepy imageio==2.4.1

# 새로 설치한 패키지를 쓰려면 런타임을 다시 시작해야 한다.
# 스스로 종료하면 Colab이 자동으로 다시 시작하므로, 메뉴에서 직접 누를 필요가 없다.
import os
os.kill(os.getpid(), 9)

In [ ]:
# [준비 2] 다시 시작된 뒤 실행. 가중치 받기 + 옛 코드 수정 + 모델 불러오기 + Drive 연결
import os, io, contextlib
os.chdir('/content/SimSwap')

# 1) 가중치 (이미 있으면 건너뜀)
if not os.path.exists('arcface_model/arcface_checkpoint.tar'):
    !wget -q -P ./arcface_model https://github.com/neuralchen/SimSwap/releases/download/1.0/arcface_checkpoint.tar
    !wget -q https://github.com/neuralchen/SimSwap/releases/download/1.0/checkpoints.zip
    !unzip -q ./checkpoints.zip -d ./checkpoints
    !wget -q -P ./parsing_model/checkpoint https://github.com/neuralchen/SimSwap/releases/download/1.0/79999_iter.pth

# 2) 얼굴 검출 모델: 노트북의 OneDrive 링크가 만료되어 insightface 공식 릴리스에서 받음
if not os.path.exists('insightface_func/models/antelope/scrfd_10g_bnkps.onnx'):
    !wget -q https://github.com/deepinsight/insightface/releases/download/v0.7/antelopev2.zip -O /content/antelopev2.zip
    !unzip -o -q /content/antelopev2.zip -d /content/antelopev2_tmp
    !mkdir -p insightface_func/models/antelope
    !cp $(find /content/antelopev2_tmp -name scrfd_10g_bnkps.onnx) insightface_func/models/antelope/

# 3) 옛 코드를 최신 환경에 맞게 수정 (Week 5 트러블슈팅)
#    - ArcFace 파일은 모델 객체 전체를 저장한 것 → PyTorch 2.6+에서 weights_only=False 필요
#    - numpy에서 없어진 np.float, np.int를 다시 연결
!sed -i 's/map_location=torch.device("cpu"))/map_location=torch.device("cpu"), weights_only=False)/' models/fs_model.py
import numpy as np
np.float = float
np.int = int

import cv2
import torch
import torch.nn.functional as F
from models.models import create_model
from options.test_options import TestOptions
from insightface_func.face_detect_crop_multi import Face_detect_crop
from util.reverse2original import reverse2wholeimage
from util.norm import SpecificNorm
from parsing_model.model import BiSeNet

# 4) 모델 불러오기
opt = TestOptions()
opt.initialize()
opt.parser.add_argument('-f')                    # Colab에서 생기는 인자 오류 방지
with contextlib.redirect_stdout(io.StringIO()):  # 긴 옵션 목록 출력 숨김
    opt = opt.parse()
opt.Arc_path = './arcface_model/arcface_checkpoint.tar'
opt.isTrain = False
opt.crop_size = 224                              # 받은 가중치(people)가 224용
crop_size = opt.crop_size

torch.nn.Module.dump_patches = True
model = create_model(opt)                        # SimSwap 본체 + ArcFace(model.netArc)
model.eval()

app = Face_detect_crop(name='antelope', root='./insightface_func/models')   # 얼굴 찾기·정렬
app.prepare(ctx_id=0, det_thresh=0.6, det_size=(640, 640))

net = BiSeNet(n_classes=19).cuda()               # 합성한 얼굴을 자연스럽게 붙이는 parsing 모델
net.load_state_dict(torch.load('./parsing_model/checkpoint/79999_iter.pth'))
net.eval()
spNorm = SpecificNorm()

# 5) Google Drive: 내 사진을 읽고, 결과를 저장 (세션이 끊겨도 남음)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/faceguard'
PHOTOS = f'{DRIVE}/photos'
RESULTS = f'{DRIVE}/week6'
os.makedirs(RESULTS, exist_ok=True)

names = ['person_a_1', 'person_a_2', 'person_a_3', 'person_a_4']
target_path = './demo_file/specific1.png'
missing = [n for n in names if not os.path.exists(f'{PHOTOS}/{n}.jpg')]
print('준비 완료' if not missing else f'Drive에 사진이 없음: {missing}')

## 도구 함수

In [ ]:
# [도구 함수] Week 5에서 쓴 함수들을 한곳에 모음
mean = torch.tensor([0.485, 0.456, 0.406], device='cuda').view(1, 3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225], device='cuda').view(1, 3, 1, 1)

# ---------- 사진 ↔ 텐서 ----------
def to_tensor(img_bgr):
    # OpenCV 사진(BGR, 0~255) → (1, 3, H, W) RGB 텐서, 0~1
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).permute(2, 0, 1).float().div(255)[None].cuda()

def save_png(x, path):
    img = (x[0].permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)
    cv2.imwrite(path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

def add_margin(img_bgr, pad=112):
    # 얼굴만 꽉 찬 224×224 사진 둘레에 검은 여백 → 검출 모델이 얼굴을 찾기 쉽게
    return cv2.copyMakeBorder(img_bgr, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))

# ---------- ArcFace 명세서 ----------
def embed(x):
    # x: (1, 3, 224, 224), RGB, 0~1. SimSwap과 같은 순서: 정규화 → 112×112 → ArcFace → 길이 1
    x = x.contiguous()   # permute로 만든 텐서는 메모리가 순서대로 놓여 있지 않아 ArcFace의 view()가 실패함
    x = (x - mean) / std
    x = F.interpolate(x, size=(112, 112))
    return F.normalize(model.netArc(x), p=2, dim=1)

def load_crop(img_bgr):
    # SimSwap이 정렬해서 잘라낸 224×224 얼굴. 얼굴을 못 찾으면 None
    found = app.get(img_bgr, crop_size)
    return None if found is None else to_tensor(found[0][0])

def arc_embed_img(img_bgr):
    # 사진 → (얼굴 찾기·정렬) → ArcFace 명세서. 얼굴을 못 찾으면 None
    face = load_crop(img_bgr)
    if face is None:
        return None
    with torch.no_grad():
        return embed(face)

# ---------- PGD (랜덤 출발 + 4가지 이동 평균) ----------
SHIFTS = [(0, 0), (0, 1), (1, 0), (1, 1)]   # 2×2 중 어느 칸이 뽑혀도 통하도록

def shift(x, dy, dx):
    return torch.roll(x, shifts=(dy, dx), dims=(2, 3))

def pgd_identity(face, epsilon, steps=30):
    alpha = 2.5 * epsilon / steps
    orig_embs = [embed(shift(face, dy, dx)).detach() for dy, dx in SHIFTS]
    # 원본(유사도 1.0, 기울기 0)에서 출발하면 움직이지 못하므로 ±epsilon 안의 임의 지점에서 출발
    adv = (face + torch.empty_like(face).uniform_(-epsilon, epsilon)).clamp(0, 1).detach()
    for _ in range(steps):
        adv = adv.detach().requires_grad_(True)
        sims = [F.cosine_similarity(embed(shift(adv, dy, dx)), orig).mean()
                for (dy, dx), orig in zip(SHIFTS, orig_embs)]
        loss = sum(sims) / len(sims)
        model.netArc.zero_grad()
        loss.backward()
        adv = adv - alpha * adv.grad.sign()          # 유사도를 낮추는 방향으로 이동
        perturbation = torch.clamp(adv - face, -epsilon, epsilon)
        adv = torch.clamp(face + perturbation, 0, 1)
    return adv.detach()

# ---------- SimSwap 합성 ----------
def swap_with_latent(latend_id, output_path):
    # 명세서를 받아 target 사진의 얼굴을 다시 그리고 저장
    img_b_whole = cv2.imread(target_path)
    crops_b, mats_b = app.get(img_b_whole, crop_size)
    tensors_b, results = [], []
    with torch.no_grad():
        for b_crop in crops_b:
            b = to_tensor(b_crop)
            results.append(model(None, b, latend_id, None, True)[0])
            tensors_b.append(b)
        reverse2wholeimage(tensors_b, results, mats_b, crop_size, img_b_whole, None,
                           output_path, no_simswaplogo=True, pasring_model=net, use_mask=True, norm=spNorm)

# ---------- 판정 (Week 5에서 정한 ArcFace 기준) ----------
ARC_FAIL = 0.406      # 진짜 내 사진끼리의 최솟값. 이 이상이면 나로 볼 수 있음
ARC_SUCCESS = 0.2     # 본인 최소와 타인(약 0)의 중간. 이 아래면 보호 성공

def judge(score):
    if score is None:
        return '검출 실패'
    if score >= ARC_FAIL:
        return '실패'
    return '부분' if score >= ARC_SUCCESS else '성공'

# ---------- 실험 한 번 ----------
def run(tag, eps, steps=30):
    # 보호 사진 만들기 → (B) 재정렬로 SimSwap 합성 → 결과 vs 내 사진 (ArcFace) 채점
    # 결과는 Drive의 week6/<tag>/ 에 저장. eps=0이면 노이즈 없이 baseline
    out_dir = f'{RESULTS}/{tag}'
    os.makedirs(out_dir, exist_ok=True)
    scores = {}
    for name in names:
        photo = cv2.imread(f'{PHOTOS}/{name}.jpg')
        me = arc_embed_img(photo)                          # 채점 기준: 원본 내 사진
        face = load_crop(photo)
        adv = face if eps == 0 else pgd_identity(face, eps / 255, steps)
        png = f'{out_dir}/{name}_eps{eps}.png'
        save_png(adv, png)                                 # 보호 얼굴 (PNG, 무손실)

        lat = arc_embed_img(add_margin(cv2.imread(png)))   # SimSwap이 다시 찾고 정렬한 명세서
        score = None
        if lat is not None:
            result_path = f'{out_dir}/{name}_eps{eps}_realign.jpg'
            swap_with_latent(lat, result_path)
            score = (arc_embed_img(cv2.imread(result_path)) * me).sum().item()
        scores[name] = score
        shown = '-' if score is None else f'{score:.3f}'
        print(f'[{tag}] {name} eps={eps:>2}/255 steps={steps} | (B) 재정렬 결과 vs 내 사진 (ArcFace) {shown} ({judge(score)})')
    return scores

print('도구 함수 준비 완료')

## 확인: Week 5 결과가 다시 나오는가

- `eps=0`: 보호 안 한 baseline → 0.68 ~ 0.77 근처 (Week 5)
- `eps=8`: 랜덤 출발 + 이동 평균 → -0.18 ~ 0.09 근처 (Week 5). 랜덤 출발 때문에 정확히 같지는 않음

In [ ]:
baseline = run('check', 0)
protected8 = run('check', 8)